In [2]:
%pip install tensorflow

  Using cached protobuf-6.33.2-cp310-abi3-win_amd64.whl.metadata (593 bytes)
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/331.9 MB 5.6 MB/s eta 0:01:00
   ---------------------------------------- 1.8/331.9 MB 4.8 MB/s eta 0:01:09
   ---------------------------------------- 2.9/331.9 MB 4.9 MB/s eta 0:01:07
   ---------------------------------------- 3.9/331.9 MB 5.1 MB/s eta 0:01:05
    --------------------------------------- 5.2/331.9 MB 5.3 MB/s eta 0:01:02
    --------------------------------------- 6.0/331.9 MB 4.9 MB/s eta 0:01:07
    --------------------------------------- 7.3/331.9 MB 5.0 MB/s eta 0:01:05
   - -------------------------------------- 8.7/331.9 MB 5.3 MB/s eta 0:01:02
   - -------------------------------------- 10.0/331.9 MB 5.3 MB/s eta 0:01:01
   - -------------------------------------- 11.0/331.9 MB 5.3 MB/s eta 0:01:01
   - -------------------------------------- 12.3/331.9 MB 5.4 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flwr 1.22.0 requires protobuf<5.0.0,>=4.21.6, but you have protobuf 6.33.2 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [4]:
# =====================
# PATH
# =====================
DATA_PATH = "https://raw.githubusercontent.com/Alfa4026/Skripsi/main/dataset/hatespeech_preprocessed.csv"  # hasil preprocessing FINAL
OUT_DIR = Path("bilstm_tokenizer")
OUT_DIR.mkdir(exist_ok=True)

# =====================
# PARAM
# =====================
MAX_VOCAB = 20000
MAX_LEN = 100


In [5]:
df = pd.read_csv(DATA_PATH)

# pastikan kolom teks
TEXT_COL = "clean_text"

# label multilabel (contoh)
LABEL_COLS = [
    "HS", "Abusive", "HS_Individual", "HS_Group",
    "HS_Religion", "HS_Race", "HS_Physical",
    "HS_Gender", "HS_Other",
    "HS_Weak", "HS_Moderate", "HS_Strong"
]

texts = df[TEXT_COL].astype(str).tolist()
labels = df[LABEL_COLS].values


In [6]:
tokenizer = Tokenizer(
    num_words=MAX_VOCAB,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(texts)


In [7]:
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

y = labels.astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (12882, 100)
y shape: (12882, 12)


In [8]:
# simpan tokenizer
with open(OUT_DIR / "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# simpan data
np.save(OUT_DIR / "X.npy", X)
np.save(OUT_DIR / "y.npy", y)

print("Tokenizer & data saved to:", OUT_DIR)


Tokenizer & data saved to: bilstm_tokenizer
